In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

project_path = '/content/drive/MyDrive/Colab Notebooks/ISU NLP'
os.chdir(project_path)


In [3]:
pip install gensim==4.3.3 wordcloud==1.9.3 nltk==3.9.1 scikit-learn==1.5.2 matplotlib==3.9.2 numpy==1.26.4 scipy==1.13.1


In [4]:
import os, re, json, logging, random
from collections import Counter
import numpy as np
import pandas as pd
from string import punctuation

import nltk
from nltk.corpus import stopwords
from nltk import sent_tokenize, word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from gensim.models.phrases import Phrases, Phraser
from wordcloud import WordCloud


logging.basicConfig(level=logging.INFO)
INPUT_JSON = "QTL_text.json"
TRAIT_DICT = "Trait_dictionary.txt"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)


random.seed(42)
np.random.seed(42)


nltk.download("punkt")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")


STOPWORDS = set(stopwords.words("english"))
PUNCT = set(punctuation)


def load_qtl(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return [r["Abstract"] for r in data if r.get("Category") == "1"]

def load_traits(path):
    with open(path, "r", encoding="utf-8") as f:
        return {line.strip().lower() for line in f if line.strip()}

def preprocess(texts):
    docs, sents = [], []
    for abs_text in texts:
        doc_tokens = []
        for sent in sent_tokenize(abs_text):
            tokens = [t.lower() for t in word_tokenize(sent)]
            tokens = [t for t in tokens if re.match(r"[a-z0-9\-]+", t)]
            tokens = [t for t in tokens if t not in STOPWORDS and t not in PUNCT]
            if tokens:
                sents.append(tokens)
                doc_tokens.extend(tokens)
        if doc_tokens:
            docs.append(doc_tokens)
    return docs, sents

def make_wordcloud(weights, filename):
    wc = WordCloud(width=800, height=800, background_color="white", random_state=42)
    wc.generate_from_frequencies(weights)
    wc.to_file(os.path.join(OUT_DIR, filename))
    logging.info("Saved %s", filename)


def task1_wordclouds(docs):
    freq = Counter([t for doc in docs for t in doc])
    make_wordcloud(freq, "wordcloud_frequency.png")

    texts = [" ".join(doc) for doc in docs]
    vec = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
    X = vec.fit_transform(texts)
    scores = np.asarray(X.mean(axis=0)).ravel()
    tfidf = dict(zip(vec.get_feature_names_out(), scores))
    make_wordcloud(tfidf, "wordcloud_tfidf.png")
    return tfidf


def task2_word2vec(sents, tfidf):
    model = Word2Vec(sentences=sents, vector_size=100, window=5, min_count=10, seed=42)
    top10 = sorted(tfidf.items(), key=lambda x: x[1], reverse=True)[:10]
    with open(os.path.join(OUT_DIR, "similar_words_raw.txt"), "w") as f:
        for word, _ in top10:
            f.write(f"== {word} ==\n")
            if word in model.wv:
                for w, s in model.wv.most_similar(word, topn=20):
                    f.write(f"{w}\t{s:.4f}\n")
            else:
                f.write("(not in vocab)\n")
            f.write("\n")
    logging.info("Saved similar_words_raw.txt")


def build_ngrams(sentences, docs, max_n=5, min_count=2, threshold=2.0):
    current_sents = sentences
    current_docs = docs
    for n in range(2, max_n + 1):
        phrases = Phrases(current_sents, min_count=min_count, threshold=threshold, delimiter="_")
        phraser = Phraser(phrases)
        current_sents = [phraser[s] for s in current_sents]
        current_docs = [phraser[d] for d in current_docs]
    return current_sents, current_docs


def task3_phrases(sents, docs, traits, max_n=5):
    sents_ngram, docs_ngram = build_ngrams(sents, docs, max_n=max_n)

    freq = Counter([t for doc in docs_ngram for t in doc])
    make_wordcloud(freq, "phrase_wordcloud_frequency.png")

    texts = [" ".join(doc) for doc in docs_ngram]
    vec = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
    X = vec.fit_transform(texts)
    scores = np.asarray(X.mean(axis=0)).ravel()
    tfidf = dict(zip(vec.get_feature_names_out(), scores))
    make_wordcloud(tfidf, "phrase_wordcloud_tfidf.png")

    model = Word2Vec(sentences=sents_ngram, vector_size=100, window=5, min_count=10, seed=42)
    top10 = sorted(tfidf.items(), key=lambda x: x[1], reverse=True)[:10]
    with open(os.path.join(OUT_DIR, "similar_words_phrases.txt"), "w") as f:
        for word, _ in top10:
            f.write(f"== {word} ==\n")
            if word in model.wv:
                for w, s in model.wv.most_similar(word, topn=20):
                    f.write(f"{w}\t{s:.4f}\n")
            else:
                f.write("(not in vocab)\n")
            f.write("\n")

    phrases = {t for doc in docs_ngram for t in doc if "_" in t}
    normalized_phrases = {p.replace("_", " ").lower() for p in phrases}
    matches = [p for p in normalized_phrases if p in traits]

    with open(os.path.join(OUT_DIR, "trait_dictionary_match_gensim.txt"), "w") as f:
        f.write(f"Total extracted phrases (up to {max_n}-gram): {len(normalized_phrases)}\n")
        f.write(f"Matches in trait dictionary: {len(matches)}\n")
        f.write("Matched terms:\n")
        for m in matches:
            f.write(m + "\n")
    logging.info("Saved trait_dictionary_match_gensim.txt")


def task4_np_chunking(sents, traits):
    grammar = r"NP: {<JJ>*<NN.*>+}"
    cp = nltk.RegexpParser(grammar)

    extracted_phrases = []
    for sent in sents:
        tagged = nltk.pos_tag(sent)
        tree = cp.parse(tagged)
        for subtree in tree.subtrees(filter=lambda t: t.label() == 'NP'):
            phrase = " ".join(word for word, pos in subtree.leaves())
            extracted_phrases.append(phrase.lower())

    unique_phrases = set(extracted_phrases)
    matches = [p for p in unique_phrases if p in traits]

    with open(os.path.join(OUT_DIR, "trait_dictionary_match_npchunk.txt"), "w") as f:
        f.write(f"Total extracted NP phrases: {len(unique_phrases)}\n")
        f.write(f"Matches in trait dictionary: {len(matches)}\n")
        f.write("Matched terms:\n")
        for m in matches:
            f.write(m + "\n")

    logging.info("Saved trait_dictionary_match_npchunk.txt")


def main():
    abstracts = load_qtl(INPUT_JSON)
    traits = load_traits(TRAIT_DICT)
    docs, sents = preprocess(abstracts)

    tfidf = task1_wordclouds(docs)
    task2_word2vec(sents, tfidf)
    task3_phrases(sents, docs, traits, max_n=5)
    task4_np_chunking(sents, traits)

if __name__ == "__main__":
    main()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
